# Reproduksi P1–P8 dengan SCIP

Jalankan sel dari atas ke bawah. Notebook dapat dijalankan dari repositori lokal atau dibuka di Google Colab. Pada Colab, kode mengambil repositori publik secara otomatis. Mulai dari P1 Low; profil Pareto menjalankan 50 optimisasi dan profil nonlinear stokastik dapat memerlukan beberapa menit.


In [ ]:
%pip install -q PySCIPOpt==6.3.0 numpy==2.3.5


In [ ]:
from pathlib import Path
import os, sys, subprocess, hashlib

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
REPOSITORY = 'https://github.com/kiittkiidd/Lampiran-Model-Matematika-Tesis.git'
REVISION = 'main'  # Ganti dengan SHA commit bila mereproduksi snapshot tertentu.
CANDIDATES = [Path.cwd() / 'source', Path.cwd() / 'math_model_acuan' / 'source']
SOURCE = next((p for p in CANDIDATES if (p / 'solve.py').exists()), None)
if SOURCE is None:
    checkout = Path.cwd() / 'Lampiran-Model-Matematika-Tesis'
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(checkout)], check=True)
    if REVISION != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', REVISION], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    SOURCE = checkout / 'math_model_acuan' / 'source'
EXPECTED = {'data.json': 'ffcbcbcc92f4e49121cbfdd11e0cbe478b612ac75c9620e16704565c14f2924a', 'scip_solver.py': 'a48dcacf5366fbaa649cd306cc091cf4b5345713ca3fbfe3a3bbde1664ad3166', 'network.py': '0510702ef413c1ddc760d68f4e349197af4177a2c129220803c7ed11edfc711e'}
for filename, expected in EXPECTED.items():
    actual = hashlib.sha256((SOURCE / filename).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'{filename} berbeda dari acuan 1.0.0-SCIP. Gunakan commit versi yang sesuai.')
sys.path.insert(0, str(SOURCE.resolve()))
from solve import solve_default
from pyscipopt import Model
m = Model()
print('SCIP:', m.getMajorVersion(), m.getMinorVersion(), m.getTechVersion())
print('Folder sumber:', SOURCE)


## Pilih profil dan tingkat

Gunakan `P1` sampai `P8`. Tingkat rute adalah `L`, `M`, atau `H`. Mulai dari P1 Low untuk memeriksa instalasi.

In [ ]:
PROFILE = 'P1'
LEVEL = 'L'

def show_step(step):
    print(f"Langkah {step['step']}: {step['goal']} | {step['status']} | {step['solver_seconds']:.4f} detik | LP {step['lp_iterations']}", flush=True)

result = solve_default(PROFILE, LEVEL, progress=show_step)
print('Biaya minimum:', result['cost_min'])
print('Gudang:', ['G' + str(g) for g in result['warehouses']])
print('Waktu solver (detik):', result['solver_seconds'])
print('Waktu total (detik):', result['total_seconds'])
print('Iterasi LP:', result['lp_iterations'])
print('Jumlah langkah optimisasi:', result['calls'])
print('Jumlah titik Pareto unik tidak didominasi:', len(result['frontier']))


## Periksa keluaran terhadap persamaan asli

Pemeriksaan ini menghitung ulang neraca, permintaan, kapasitas, waktu, biaya, dan emisi langsung dari kiriman, tanpa memakai matriks internal solver.

In [ ]:
from verify import check_record
from scip_solver import DATA
print(check_record(result, DATA))


## Simpan hasil

Biaya dan emisi dihitung ulang dari kiriman asli. File hasil juga menyimpan waktu dan penghitung kerja per langkah.

In [ ]:
import json
path = Path(f'hasil_{PROFILE}_{LEVEL}.json')
path.write_text(json.dumps(result, indent=2))
print(path.resolve())


## Menjalankan ulang benchmark lengkap

Benchmark lengkap menjalankan 24 kombinasi profil–tingkat sebanyak tiga pengukuran setelah pemanasan. Gunakan salinan folder sumber untuk pengukuran baru agar hasil versi yang dibekukan tetap tersimpan. Urutan pekerjaan diacak menggunakan seed yang dicatat di konfigurasi. Seluruh hasil mentah yang dipakai dalam laporan tersedia di folder `math_model_acuan/hasil` repositori.

Perintah dari terminal: `OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python source/benchmark.py`. Benchmark nonlinear stokastik dapat berlangsung lama. Iterasi LP dan iterasi NLP merupakan dua penghitung berbeda; keduanya tersedia dalam hasil per langkah.